<a href="https://colab.research.google.com/github/AnnaPaulaFigueiredo/data_master/blob/main/01_DM_CASE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
from google.colab import drive
import warnings
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import shutil
import warnings
from pyspark.sql import functions as F
import shutil
from pyspark.sql import DataFrame

# Desativa os avisos em uma linha própria
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option('display.max_columns', 500)
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import NumericType, DateType, TimestampType, DoubleType, FloatType
import humanize
from pyspark.sql.functions import to_date, col, concat, lit
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, DateType, TimestampType, StringType
if 'spark' not in locals():
    spark = SparkSession.builder.appName("DM_CASE").getOrCreate()
import pandas as pd
import shutil
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, to_date
import shutil
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import shutil
from pyspark.sql import functions as F
from pyspark.sql import Window
pd.set_option('display.max_columns', None)

ids = [
    "++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=",
    "+++EI4HgyhgcJHIPXk/VRP7bt17+2joG39T6oEfJ+tc=",
    "+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=",
    "+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=",
    "+++4vcS9aMH7KWdfh5git6nA5fC5jjisd5H/NcM++WM=",
    "++/TR7WI15q2ZCtOXmoap7jR+kEhbMVE5swOqsfqpqI=",
    "+++4vcS9aMH7KWdfh5git6nA5fC5jjisd5H/NcM++WM="
]

# Global vars

In [2]:
IDENTIFY = "msno"
TIME_COL = "safra"

output_analysis_path = "/content/drive/MyDrive/SANTANDER/merged_ids_analysis.parquet"
dfs_members = spark.read.parquet("/content/drive/MyDrive/SANTANDER/members.parquet")
dfs_transactions = spark.read.parquet("/content/drive/MyDrive/SANTANDER/transactions.parquet")
dfs_logs = spark.read.parquet("/content/drive/MyDrive/SANTANDER/user_logs.parquet")

# Functions

## Data Quality

In [ ]:
def void_input(df, col, value):
    return df.withColumn(
        col,
        F.when(F.col(col).isNull(), value).otherwise(F.col(col))
    )

def is_numeric_string(sample):
    try:
        return all(float(x) or True for x in sample if x is not None)
    except:
        return False

def analyse_columns(df_spark):
    resultado = []
    total_linhas = df_spark.count()

    for campo in df_spark.schema.fields:
        nome = campo.name
        tipo = campo.dataType

        unicos_sample = (
            df_spark.select(nome)
            .distinct()
            .limit(10)
            .toPandas()[nome]
            .tolist()
        )

        n_valores_unicos = df_spark.select(nome).distinct().count()

        if isinstance(tipo, NumericType):
            tipo_base = "Numérico"

        elif isinstance(tipo, (DateType, TimestampType)):
            tipo_base = "Data"

        elif isinstance(tipo, StringType):
            if is_numeric_string(unicos_sample):
                tipo_base = "Numérico (string)"
            else:
                tipo_base = "Categórico"

        else:
            tipo_base = "Outro"


        if isinstance(tipo, NumericType):
            nulos = df_spark.filter(
                F.col(nome).isNull() | F.isnan(F.col(nome))
            ).count()
        else:
            nulos = df_spark.filter(F.col(nome).isNull()).count()

        duplicados = (
            df_spark.groupBy(nome)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )


        eh_binario = n_valores_unicos == 2


        eh_ordinal = False

        if tipo_base.startswith("Numérico") and 2 < n_valores_unicos <= 10:
            eh_ordinal = True

        elif tipo_base == "Categórico":
            if n_valores_unicos <= 6:
                eh_ordinal = True

        if tipo_base.startswith("Numérico") or tipo_base == "Data":
            stats = df_spark.select(
                F.min(F.col(nome)).alias("minimo"),
                F.max(F.col(nome)).alias("maximo")
            ).first()

            minimo = stats["minimo"]
            maximo = stats["maximo"]
        else:
            minimo = None
            maximo = None

        resultado.append({
            "coluna": nome,
            "tipo_base": tipo_base,
            "ordinal": eh_ordinal,
            "binário": eh_binario,
            "mínimo": minimo,
            "máximo": maximo,
            "nulos": nulos,
            "duplicados": duplicados,
            "n° valores únicos": n_valores_unicos,
            "valores únicos (sample)": unicos_sample[:5] + (["..."] if len(unicos_sample) > 5 else [])
        })

    return pd.DataFrame(resultado)


def cast_columns(
    df,
    cols_int=None,
    cols_float=None,
    cols_string=None,
    cols_date=None,
    date_format="yyyy-MM-dd"
):
    """
    Converte colunas de um DataFrame para tipos específicos.

    Parâmetros:
    - df: DataFrame Spark
    - cols_int: lista de colunas para int
    - cols_float: lista de colunas para float/double
    - cols_string: lista de colunas para string
    - cols_date: lista de colunas para date
    - date_format: formato das datas (default: yyyy-MM-dd)

    Retorna:
    - DataFrame com casts aplicados
    """

    cols_int = cols_int or []
    cols_float = cols_float or []
    cols_string = cols_string or []
    cols_date = cols_date or []

    df_out = df

    # Inteiro
    for c in cols_int:
        df_out = df_out.withColumn(c, col(c).cast("int"))

    # Float / Double
    for c in cols_float:
        df_out = df_out.withColumn(c, col(c).cast("double"))

    # String
    for c in cols_string:
        df_out = df_out.withColumn(c, col(c).cast("string"))

    # Date
    for c in cols_date:
        df_out = df_out.withColumn(c, to_date(col(c), date_format))

    return df_out

## Examples

In [ ]:
%%time
dfs_members = spark.read.parquet("/content/drive/MyDrive/SANTANDER/members.parquet")
dfs_members.filter((F.col('msno') == ids[0])).orderBy(F.col("safra")).show(truncate=False)
df_stats = analyse_columns(dfs_members)
df_stats

+--------------------------------------------+------+----------------------+----+---+------+--------------+--------+
|msno                                        |safra |registration_init_time|city|bd |gender|registered_via|is_ativo|
+--------------------------------------------+------+----------------------+----+---+------+--------------+--------+
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201601|20140421              |13  |39 |male  |3             |1       |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201602|20140421              |13  |39 |male  |3             |1       |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201603|20140421              |13  |39 |male  |3             |1       |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201604|20140421              |13  |39 |male  |3             |1       |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201605|20140421              |13  |39 |male  |3             |1       |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201606|20140421   

,coluna,tipo_base,ordinal,binário,mínimo,máximo,nulos,duplicados,n° valores únicos,valores únicos (sample)
0,msno,Categórico,False,False,None,None,0,6114345,6287789,"[++7jKYbuIJPXry8Oh1NcEh9fCsqcQgUaaxXsgG15kMg=,..."
1,safra,Numérico (string),False,False,201601,201612,0,12,12,"[201607, 201611, 201605, 201602, 201606, ...]"
2,registration_init_time,Numérico (string),False,False,20040326,20161231,0,4663,4663,"[20150729, 20120909, 20120322, 20160429, 20141..."
3,city,Numérico (string),False,False,1,9,0,21,21,"[7, 15, 11, 3, 8, ...]"
4,bd,Numérico (string),False,False,-10,994,0,381,385,"[944, 51, 7, -489, 124, ...]"
5,gender,Categórico,True,False,None,None,38210177,3,3,"[female, male, None]"
6,registered_via,Numérico (string),False,False,-1,9,0,17,17,"[7, 11, 3, 8, 16, ...]"
7,is_ativo,Numérico,False,True,0,1,0,2,2,"[1, 0]"


In [ ]:
%%time
dfs_transactions = spark.read.parquet("/content/drive/MyDrive/SANTANDER/transactions.parquet")
dfs_transactions.filter((F.col('msno') == ids[0])).orderBy(F.col("safra")).show(truncate=False)
df_stats = analyse_columns(dfs_transactions)
df_stats

+--------------------------------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+------+
|msno                                        |payment_method_id|payment_plan_days|plan_list_price|actual_amount_paid|is_auto_renew|transaction_date|membership_expire_date|is_cancel|safra |
+--------------------------------------------+-----------------+-----------------+---------------+------------------+-------------+----------------+----------------------+---------+------+
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|14               |0                |0              |149               |1            |20150331        |20150430              |0        |201503|
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|14               |0                |0              |149               |1            |20150630        |20150731              |0        |201506|
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|14       

,coluna,tipo_base,ordinal,binário,mínimo,máximo,nulos,duplicados,n° valores únicos,valores únicos (sample)
0,msno,Categórico,False,False,None,None,0,1697963,2363626,"[++4RuqBw0Ss6bQU4oMxaRlbBPoWzoEiIZaxPM04Y4+U=,..."
1,payment_method_id,Numérico (string),False,False,1,8,0,40,40,"[7, 15, 11, 29, 3, ...]"
2,payment_plan_days,Numérico (string),False,False,0,99,0,34,37,"[7, 15, 200, 3, 30, ...]"
3,plan_list_price,Numérico (string),False,False,0,99,0,49,51,"[447, 124, 15, 894, 1788, ...]"
4,actual_amount_paid,Numérico (string),False,False,0,99,0,51,57,"[447, 124, 15, 894, 1788, ...]"
5,is_auto_renew,Numérico (string),False,True,0,1,0,2,2,"[0, 1]"
6,transaction_date,Numérico (string),False,False,20150101,20170228,0,790,790,"[20160615, 20160820, 20161119, 20160825, 20150..."
7,membership_expire_date,Numérico (string),False,False,19700101,20170331,0,1473,1534,"[20160615, 20161119, 20160429, 20160820, 20160..."
8,is_cancel,Numérico (string),False,True,0,1,0,2,2,"[0, 1]"
9,safra,Numérico,False,False,201501,201702,0,26,26,"[201505, 201701, 201702, 201501, 201509, ...]"


In [ ]:
%%time
dfs_log = spark.read.parquet("/content/drive/MyDrive/SANTANDER/user_logs.parquet")
dfs_log.filter((F.col('msno') == ids[0])).orderBy(F.col("safra")).show(truncate=False)
df_stats = analyse_columns(dfs_log)
df_stats

+--------------------------------------------+------+------+------+------+-------+-------+-------+------------------+
|msno                                        |safra |num_25|num_50|num_75|num_985|num_100|num_unq|total_secs        |
+--------------------------------------------+------+------+------+------+-------+-------+-------+------------------+
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201501|7.0   |1.0   |3.0   |2.0    |63.0   |60.0   |16674.686         |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201502|32.0  |10.0  |5.0   |8.0    |487.0  |328.0  |116105.431        |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201503|6.0   |2.0   |2.0   |1.0    |139.0  |138.0  |34775.509         |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201504|5.0   |4.0   |1.0   |0.0    |169.0  |168.0  |43534.784         |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201505|3.0   |0.0   |2.0   |0.0    |25.0   |29.0   |6340.997          |
|++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=|201506|0.0

,coluna,tipo_base,ordinal,binário,mínimo,máximo,nulos,duplicados,n° valores únicos,valores únicos (sample)
0,msno,Categórico,False,False,NaN,NaN,0,2293336,5234111,"[21Ld7rZnJm9Sa6+Lc7JVsvAbdX2F8g7dIe8vIqs4B6s=,..."
1,safra,Numérico,False,False,2.015010e+05,2.017020e+05,0,26,26,"[201505, 201701, 201702, 201501, 201509, ...]"
2,num_25,Numérico,False,False,0.000000e+00,1.118640e+05,0,3816,4891,"[596.0, 299.0, 305.0, 692.0, 1051.0, ...]"
3,num_50,Numérico,False,False,0.000000e+00,8.875000e+03,0,1233,1587,"[299.0, 305.0, 558.0, 496.0, 769.0, ...]"
4,num_75,Numérico,False,False,0.000000e+00,3.485000e+03,0,746,986,"[299.0, 305.0, 596.0, 170.0, 147.0, ...]"
5,num_985,Numérico,False,False,0.000000e+00,3.769800e+04,0,1740,2509,"[299.0, 596.0, 305.0, 558.0, 769.0, ...]"
6,num_100,Numérico,False,False,0.000000e+00,1.967410e+05,0,10573,13047,"[305.0, 299.0, 692.0, 596.0, 2815.0, ...]"
7,num_unq,Numérico,False,False,1.000000e+00,3.270600e+04,0,7450,8593,"[934.0, 305.0, 596.0, 496.0, 558.0, ...]"
8,total_secs,Numérico,False,False,-2.398077e+17,9.223372e+15,0,1395426,24025199,"[219986.77300000002, 121822.54300000002, 80922..."


# Check joins

O objetivo é manter nas bases, só os clientes que tem dados nas três tabelas, por completo.

Para otimizar os cálculos a priori.

## Ids Analysis

In [ ]:
%%time
def adicionar_meses(ano, mes, delta):
    """
    Adiciona ou subtrai meses de uma combinação ano/mês.

    Exemplo:
        adicionar_meses(2016, 1, -3)
        -> (2015, 10)
    """

    indice = ano * 12 + (mes - 1) + delta

    novo_ano = indice // 12
    novo_mes = indice % 12 + 1

    return novo_ano, novo_mes


# ============================================================
# ANALISAR BASES E IDENTIFICAR PÚBLICO-ALVO
# ============================================================

def analisar_bases(
    df1: DataFrame,
    df2: DataFrame,
    df3: DataFrame,
    nome1="base1",
    nome2="base2",
    nome3="base3",
    safra_modelagem_inicial="201601",
    meses_historico=3
):
    """
    Identifica o público-alvo por cliente e safra.

    Um cliente é elegível em determinada safra quando:

        1. Está presente nas três bases naquela safra;
        2. Possui os N meses imediatamente anteriores em TRANSACTIONS;
        3. Possui os N meses imediatamente anteriores em LOGS.

    Exemplo com 3 meses de histórico:

        Safra 201601
            histórico: 201510, 201511, 201512

        Safra 201602
            histórico: 201511, 201512, 201601

        Safra 201603
            histórico: 201512, 201601, 201602

    IMPORTANTE:
        A janela histórica é móvel e calculada individualmente
        para cada safra de modelagem.
    """

    print("=" * 70)
    print("📊 ANÁLISE DE INTERSECÇÃO DAS BASES")
    print("=" * 70)

    # ========================================================
    # 1. DOCUMENTAÇÃO DA PRIMEIRA SAFRA
    # ========================================================

    ano_modelagem = int(safra_modelagem_inicial[:4])
    mes_modelagem = int(safra_modelagem_inicial[4:])

    safras_historico_inicial = []

    for i in range(meses_historico, 0, -1):

        ano, mes = adicionar_meses(
            ano_modelagem,
            mes_modelagem,
            -i
        )

        safras_historico_inicial.append(
            f"{ano:04d}{mes:02d}"
        )

    print(
        f"📅 Primeira safra de modelagem : "
        f"{safra_modelagem_inicial}"
    )

    print(
        f"📅 Histórico mínimo necessário : "
        f"{meses_historico} meses"
    )

    print(
        f"📅 Para {safra_modelagem_inicial}, "
        f"histórico: {safras_historico_inicial}"
    )

    # ========================================================
    # 2. PADRONIZA SAFRA COMO DATE
    # ========================================================

    df1_base = (
        df1
        .select(
            "msno",
            F.to_date(
                F.col("safra").cast("string"),
                "yyyyMM"
            ).alias("safra_date")
        )
        .filter(F.col("safra_date").isNotNull())
        .distinct()
    )

    df2_base = (
        df2
        .select(
            "msno",
            F.to_date(
                F.col("safra").cast("string"),
                "yyyyMM"
            ).alias("safra_date")
        )
        .filter(F.col("safra_date").isNotNull())
        .distinct()
    )

    df3_base = (
        df3
        .select(
            "msno",
            F.to_date(
                F.col("safra").cast("string"),
                "yyyyMM"
            ).alias("safra_date")
        )
        .filter(F.col("safra_date").isNotNull())
        .distinct()
    )

    # ========================================================
    # 3. CLIENTES PRESENTES NAS TRÊS BASES
    #
    # Aqui a presença também é considerada por SAFRA.
    # ========================================================

    candidatos = (
        df1_base
        .join(
            df2_base,
            on=["msno", "safra_date"],
            how="inner"
        )
        .join(
            df3_base,
            on=["msno", "safra_date"],
            how="inner"
        )
        .withColumn(
            "presenca",
            F.lit("111")
        )
    )

    # ========================================================
    # 4. CRIA AS SAFRAS HISTÓRICAS NECESSÁRIAS
    #
    # Para cada msno + safra:
    #
    # 201602
    #    ↓
    # 201511
    # 201512
    # 201601
    #
    # 201603
    #    ↓
    # 201512
    # 201601
    # 201602
    # ========================================================

    candidatos_historico = (
        candidatos
        .withColumn(
            "safras_historico",
            F.expr(
                f"""
                transform(
                    sequence(
                        1,
                        {meses_historico}
                    ),
                    x -> add_months(
                        safra_date,
                        -x
                    )
                )
                """
            )
        )
        .withColumn(
            "safra_historico",
            F.explode("safras_historico")
        )
        .drop("safras_historico")
    )

    # ========================================================
    # 5. VERIFICA HISTÓRICO EM TRANSACTIONS
    # ========================================================

    transactions_historico = (
        df2_base
        .withColumnRenamed("safra_date", "safra_historico") # Corrigido
        .withColumn(
            "tem_transactions",
            F.lit(1)
        )
    )

    hist_transactions = (
        candidatos_historico
        .join(
            transactions_historico,
            on=["msno", "safra_historico"],
            how="left"
        )
        .groupBy(
            "msno",
            "safra_date"
        )
        .agg(
            F.count(
                F.col("tem_transactions")
            ).alias(
                "qtd_meses_historico_transactions"
            )
        )
    )

    # ========================================================
    # 6. VERIFICA HISTÓRICO EM LOGS
    # ========================================================

    logs_historico = (
        df3_base
        .withColumnRenamed("safra_date", "safra_historico") # Corrigido
        .withColumn(
            "tem_logs",
            F.lit(1)
        )
    )

    hist_logs = (
        candidatos_historico
        .join(
            logs_historico,
            on=["msno", "safra_historico"],
            how="left"
        )
        .groupBy(
            "msno",
            "safra_date"
        )
        .agg(
            F.count(
                F.col("tem_logs")
            ).alias(
                "qtd_meses_historico_logs"
            )
        )
    )

    # ========================================================
    # 7. JUNTA AS INFORMAÇÕES DE HISTÓRICO
    # ========================================================

    merged = (
        candidatos

        .join(
            hist_transactions,
            on=["msno", "safra_date"],
            how="left"
        )

        .join(
            hist_logs,
            on=["msno", "safra_date"],
            how="left"
        )

        .fillna({
            "qtd_meses_historico_transactions": 0,
            "qtd_meses_historico_logs": 0
        })
    )

    # ========================================================
    # 8. CONVERTE SAFRA PARA YYYYMM
    # ========================================================

    merged = merged.withColumn(
        "safra",
        F.date_format(
            F.col("safra_date"),
            "yyyyMM"
        )
    )

    # ========================================================
    # 9. VALIDA HISTÓRICO COMPLETO
    # ========================================================

    merged = merged.withColumn(
        "historico_completo_transactions",
        F.when(
            F.col(
                "qtd_meses_historico_transactions"
            ) >= meses_historico,
            1
        ).otherwise(0)
    )

    merged = merged.withColumn(
        "historico_completo_logs",
        F.when(
            F.col(
                "qtd_meses_historico_logs"
            ) >= meses_historico,
            1
        ).otherwise(0)
    )

    # ========================================================
    # 10. DEFINE PÚBLICO-ALVO
    # ========================================================

    merged = merged.withColumn(
        "publico_alvo",
        F.when(
            (F.col("presenca") == "111") &
            (
                F.col(
                    "historico_completo_transactions"
                ) == 1
            ) &
            (
                F.col(
                    "historico_completo_logs"
                ) == 1
            ),
            1
        ).otherwise(0)
    )

    # ========================================================
    # 11. EXPLICA O MOTIVO DA ELEGIBILIDADE
    # ========================================================

    merged = merged.withColumn(
        "status_publico",

        F.when(
            F.col("historico_completo_transactions") == 0,
            "Sem histórico completo em transactions"
        )

        .when(
            F.col("historico_completo_logs") == 0,
            "Sem histórico completo em logs"
        )

        .otherwise(
            "Público-alvo"
        )
    )

    # ========================================================
    # 12. ORGANIZA COLUNAS
    # ========================================================

    colunas = [
        "msno",
        "safra",

        "presenca",

        "qtd_meses_historico_transactions",
        "historico_completo_transactions",

        "qtd_meses_historico_logs",
        "historico_completo_logs",

        "publico_alvo",
        "status_publico"
    ]

    merged = merged.select(colunas)

    # ========================================================
    # 13. RESUMO POR SAFRA
    # ========================================================

    print()
    print("=" * 70)
    print("📊 ELEGIBILIDADE POR SAFRA")
    print("=" * 70)

    resumo = (
        merged
        .groupBy("safra", "publico_alvo")
        .agg(
            F.countDistinct("msno").alias(
                "qtd_clientes"
            )
        )
        .orderBy(
            "safra",
            "publico_alvo"
        )
    )

    resumo.show(truncate=False)

    # ========================================================
    # 14. RESUMO DO PÚBLICO-ALVO
    # ========================================================

    print()
    print("=" * 70)
    print("🎯 RESUMO DO PÚBLICO-ALVO")
    print("=" * 70)

    resumo_publico = (
        merged
        .groupBy(
            "safra",
            "status_publico"
        )
        .agg(
            F.countDistinct("msno").alias(
                "qtd_clientes"
            )
        )
        .orderBy(
            "safra",
            "status_publico"
        )
    )

    resumo_publico.show(truncate=False)

    # ========================================================
    # 15. RETORNO
    # ========================================================

    return merged, resumo, resumo_publico

merged_ids, resumo, resumo_publico = analisar_bases(
    dfs_members,
    dfs_transactions,
    dfs_logs,
    nome1="members",
    nome2="transactions",
    nome3="logs",
    safra_modelagem_inicial="201601",
    meses_historico=3
)

📊 ANÁLISE DE INTERSECÇÃO DAS BASES
📅 Primeira safra de modelagem : 201601
📅 Histórico mínimo necessário : 3 meses
📅 Para 201601, histórico: ['201510', '201511', '201512']

📊 ELEGIBILIDADE POR SAFRA
+------+------------+------------+
|safra |publico_alvo|qtd_clientes|
+------+------------+------------+
|201601|0           |270385      |
|201601|1           |376725      |
|201602|0           |200744      |
|201602|1           |398075      |
|201603|0           |188848      |
|201603|1           |405053      |
|201604|0           |174595      |
|201604|1           |420866      |
|201605|0           |166403      |
|201605|1           |439530      |
|201606|0           |165061      |
|201606|1           |455270      |
|201607|0           |258105      |
|201607|1           |463250      |
|201608|0           |273655      |
|201608|1           |474992      |
|201609|0           |276251      |
|201609|1           |483174      |
|201610|0           |215800      |
|201610|1           |568488     

In [ ]:
publico_alvo = (
    merged_ids
    .filter(F.col("publico_alvo") == 1)
)

qtd_elegiveis = publico_alvo.select(IDENTIFY).distinct().count()

print(f"👥 Clientes elegíveis: {qtd_elegiveis:,}")

shutil.rmtree(output_analysis_path, ignore_errors=True)

(
    publico_alvo
    .write
    .mode("overwrite")
    .parquet(output_analysis_path)
)

print(f"💾 Público-alvo salvo em:")
print(f"   {output_analysis_path}")
print("✅ Apenas clientes elegíveis foram salvos!")

👥 Clientes elegíveis: 790,075
💾 Público-alvo salvo em:
   /content/drive/MyDrive/SANTANDER/merged_ids_analysis.parquet
✅ Apenas clientes elegíveis foram salvos!


In [ ]:
# Ativa a tolerância a falhas para arquivos corrompidos no Drive
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")

ids_commons = spark.read.parquet(output_analysis_path)
ids_commons.show(10, truncate=False)

+--------------------------------------------+------+--------+--------------------------------+-------------------------------+------------------------+-----------------------+------------+--------------+
|msno                                        |safra |presenca|qtd_meses_historico_transactions|historico_completo_transactions|qtd_meses_historico_logs|historico_completo_logs|publico_alvo|status_publico|
+--------------------------------------------+------+--------+--------------------------------+-------------------------------+------------------------+-----------------------+------------+--------------+
|++CzVs35NSfU/gjDrGbWLVMweubEiMZsuvd3eTbXs58=|201607|111     |3                               |1                              |3                       |1                      |1           |Público-alvo  |
|++Ll4H/bT6Iiq/+0xcVwubXnuf0XR0MWVh2AnHAgxa8=|201612|111     |3                               |1                              |3                       |1                      |1   

# Membros

##  

In [3]:
def preparar_members(df):

    print("=" * 70)
    print("ETAPA 1 - PREPARAÇÃO DOS MEMBERS")
    print("=" * 70)

    # --------------------------------------------------------
    # Padronização dos tipos
    # --------------------------------------------------------

    df = (
        df
        .withColumn(IDENTIFY, F.col(IDENTIFY).cast("string"))
        .withColumn("safra", F.col("safra").cast("string"))
        .withColumn("is_ativo", F.col("is_ativo").cast("integer"))
    )

    # --------------------------------------------------------
    # Data de cadastro
    # --------------------------------------------------------

    df = df.withColumn(
        "registration_init_date",
        F.to_date(
            F.col("registration_init_time").cast("string"),
            "yyyyMMdd"
        )
    )

    # --------------------------------------------------------
    # Tratamento de gender
    # --------------------------------------------------------

    df = df.withColumn(
        "gender",
        F.coalesce(
            F.col("gender"),
            F.lit("unknown")
        )
    )

    # --------------------------------------------------------
    # Data da safra
    # --------------------------------------------------------

    df = df.withColumn(
        "safra_date_dt",
        F.to_date(
            F.concat(
                F.col("safra"),
                F.lit("01")
            ),
            "yyyyMMdd"
        )
    )

    # ========================================================
    # CONSISTÊNCIA TEMPORAL
    # ========================================================

    print("\n" + "-" * 70)
    print("CONSISTÊNCIA TEMPORAL")
    print("-" * 70)

    # Antes
    qtd_antes = df.count()
    clientes_antes = df.select(IDENTIFY).distinct().count()

    print("Antes do filtro:")
    print(f"  Registros          : {qtd_antes:,}")
    print(f"  Clientes distintos : {clientes_antes:,}")

    # Filtro
    df = df.filter(
        F.col("registration_init_date")
        <=
        F.col("safra_date_dt")
    )

    # Depois
    qtd_depois = df.count()
    clientes_depois = df.select(IDENTIFY).distinct().count()

    print("\nDepois do filtro:")
    print(f"  Registros          : {qtd_depois:,}")
    print(f"  Clientes distintos : {clientes_depois:,}")

    # Diferença
    qtd_removidos = qtd_antes - qtd_depois
    clientes_removidos = clientes_antes - clientes_depois

    print("\nRemovidos pelo filtro:")
    print(f"  Registros          : {qtd_removidos:,}")
    print(f"  Clientes distintos : {clientes_removidos:,}")
    print(
        f"  % registros removidos: "
        f"{qtd_removidos / qtd_antes * 100:.2f}%"
    )

    # --------------------------------------------------------
    # Resultado final
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print("RESULTADO DA PREPARAÇÃO")
    print("-" * 70)
    print(f"Registros finais : {qtd_depois:,}")
    print(f"Colunas          : {len(df.columns)}")

    print("\nSchema:")
    df.printSchema()

    return df

##  Target

In [4]:
def criar_target_churn(df):

    print("=" * 70)
    print("ETAPA 2 - CRIAÇÃO DA TARGET DE CHURN")
    print("=" * 70)

    # ============================================================
    # M0
    # ============================================================
    #
    # Mantemos registration_init_date porque ela será utilizada
    # posteriormente para criar tempo_cliente_meses.
    #
    # Não estamos usando nenhuma informação futura aqui.

    df_m0 = (
        df.select(
            IDENTIFY,
            "safra",
            "safra_date_dt",
            "registration_init_date",
            "is_ativo"
        )
        .withColumn(
            "is_ativo_m0",
            F.col("is_ativo")
        )
        .withColumn(
            "safra_date_m3",
            F.add_months(
                F.col("safra_date_dt"),
                3
            )
        )
        .drop("is_ativo")
    )

    # ============================================================
    # M+3
    # ============================================================
    #
    # Para o target, precisamos somente do status em M+3.

    df_m3 = (
        df.select(
            IDENTIFY,
            F.col("safra_date_dt").alias("safra_date_m3"),
            F.col("is_ativo").alias("is_ativo_m3")
        )
    )

    # ============================================================
    # JOIN M0 → M+3
    # ============================================================

    df_target = df_m0.join(
        df_m3,
        on=[
            IDENTIFY,
            "safra_date_m3"
        ],
        how="left"
    )

    # ============================================================
    # TARGET
    # ============================================================

    df_target = df_target.withColumn(
        "churn_m3",
        F.when(
            (F.col("is_ativo_m0") == 1) &
            (F.col("is_ativo_m3") == 0),
            1
        )
        .when(
            (F.col("is_ativo_m0") == 1) &
            (F.col("is_ativo_m3") == 1),
            0
        )
    )

    return df_target

##  Público Alvo

In [5]:
def criar_publico_alvo(
    df_target,
    safra_inicio="201601",
    safra_fim="201609"
):

    print("=" * 70)
    print("ETAPA 3 - PÚBLICO-ALVO")
    print("=" * 70)

    df_publico = (
        df_target
        .filter(
            (F.col("is_ativo_m0") == 1)
            &
            (F.col("safra") >= safra_inicio)
            &
            (F.col("safra") <= safra_fim)
            &
            F.col("is_ativo_m3").isNotNull()
        )
    )

    print(f"Safras consideradas: {safra_inicio} até {safra_fim}")
    print(
        f"Observações cliente × safra: "
        f"{df_publico.count():,}"
    )

    print(
        f"Clientes distintos: "
        f"{df_publico.select(IDENTIFY).distinct().count():,}"
    )

    print("\nDistribuição da target:")

    (
        df_publico
        .groupBy("churn_m3")
        .count()
        .orderBy("churn_m3")
        .show()
    )

    return df_publico

##  Features

In [6]:
def adicionar_tempo_cliente_meses(df):

    df = df.withColumn(
        "tempo_cliente_meses",
        F.when(
            F.col("registration_init_date").isNull(),
            None
        ).otherwise(
            F.floor(
                F.months_between(
                    F.col("safra_date_dt"),
                    F.col("registration_init_date")
                )
            ).cast("int") + 1
        )
    )

    # Evita valores menores que 1
    df = df.withColumn(
        "tempo_cliente_meses",
        F.when(
            F.col("tempo_cliente_meses") < 1,
            1
        ).otherwise(
            F.col("tempo_cliente_meses")
        )
    )

    return df

##  Pipeline Pub Alvo

In [7]:
def get_pipeline_members(
    df: DataFrame,
    safra_inicio="201601",
    safra_fim="201609"
) -> DataFrame:
    """
    ============================================================
    PIPELINE MEMBERS
    ============================================================

    Fluxo:

    1. Preparação estrutural
    2. Criação da target M+3
    3. Definição do público-alvo

    As features dos Members serão construídas posteriormente.
    """

    df_preparado = preparar_members(df)

    df_target = criar_target_churn(
        df_preparado
    )

    df_publico = criar_publico_alvo(
        df_target,
        safra_inicio="201601",
        safra_fim="201609"
    )

    df_publico = adicionar_tempo_cliente_meses(
        df_publico
    )

    print("\n" + "=" * 70)
    print("PIPELINE MEMBERS FINALIZADO")
    print("=" * 70)

    return df_publico

In [8]:
%%time
print("=" * 60)
print("PIPELINE-MEMBERS")
print("=" * 60)

input_path = "/content/drive/MyDrive/SANTANDER/members.parquet"

df = spark.read.parquet(input_path)
'''
SEED = 42
SAMPLE_FRACTION = 0.10

fractions = {
        row["safra"]: SAMPLE_FRACTION
        for row in df.select("safra").distinct().collect()
    }

df = df.sampleBy(
        col="safra",
        fractions=fractions,
        seed=SEED
    )
'''
print(f"Registros: {df.count():,}")
print(f"Colunas: {len(df.columns)}")
dfs_members = get_pipeline_members(    df,
    safra_inicio="201601",
    safra_fim="201609"
)


PIPELINE-MEMBERS
Registros: 63,867,246
Colunas: 8
ETAPA 1 - PREPARAÇÃO DOS MEMBERS

----------------------------------------------------------------------
CONSISTÊNCIA TEMPORAL
----------------------------------------------------------------------
Antes do filtro:
  Registros          : 63,867,246
  Clientes distintos : 6,287,789

Depois do filtro:
  Registros          : 61,696,214
  Clientes distintos : 6,120,195

Removidos pelo filtro:
  Registros          : 2,171,032
  Clientes distintos : 167,594
  % registros removidos: 3.40%

----------------------------------------------------------------------
RESULTADO DA PREPARAÇÃO
----------------------------------------------------------------------
Registros finais : 61,696,214
Colunas          : 10

Schema:
root
 |-- msno: string (nullable = true)
 |-- safra: string (nullable = true)
 |-- registration_init_time: string (nullable = true)
 |-- city: string (nullable = true)
 |-- bd: string (nullable = true)
 |-- gender: string (nullable = f

In [9]:
dfs_members.groupBy("safra", "churn_m3").agg(F.countDistinct("msno").alias("count_distinct_msno")).orderBy("safra").show()

+------+--------+-------------------+
| safra|churn_m3|count_distinct_msno|
+------+--------+-------------------+
|201601|       1|             174276|
|201601|       0|             678578|
|201602|       1|             178666|
|201602|       0|             709881|
|201603|       0|             716851|
|201603|       1|             129817|
|201604|       1|              85349|
|201604|       0|             732112|
|201605|       0|             735031|
|201605|       1|              96795|
|201606|       0|             731677|
|201606|       1|              93747|
|201607|       1|             102644|
|201607|       0|             842603|
|201608|       1|             120482|
|201608|       0|             839272|
|201609|       1|             126732|
|201609|       0|             849990|
+------+--------+-------------------+



### Consistência

In [10]:
def analisar_continuidade_historico(df: DataFrame) -> DataFrame:
    """
    ============================================================
    ANÁLISE DE CONTINUIDADE DO HISTÓRICO
    ============================================================

    Objetivo:
    ----------
    Avaliar a qualidade temporal do histórico de cada cliente.

    IMPORTANTE:
    ----------
    Esta análise é DIAGNÓSTICA.

    Uma lacuna no histórico não elimina automaticamente
    o cliente do público de modelagem.

    A continuidade será considerada posteriormente apenas
    quando uma feature específica exigir uma janela completa.
    """

    print("=" * 70)
    print("ANÁLISE DE CONTINUIDADE DO HISTÓRICO")
    print("=" * 70)

    historico = (
        df
        .groupBy(IDENTIFY)
        .agg(
            F.min("safra_date_dt").alias("primeira_safra"),
            F.max("safra_date_dt").alias("ultima_safra"),
            F.countDistinct("safra_date_dt").alias("qtd_safras")
        )
    )

    # Quantidade de meses que deveriam existir entre
    # a primeira e a última safra, considerando inclusive
    # os meses inicial e final.
    historico = historico.withColumn(
        "qtd_safras_esperadas",
        (
            F.year("ultima_safra") * 12
            + F.month("ultima_safra")
            -
            (
                F.year("primeira_safra") * 12
                + F.month("primeira_safra")
            )
            + 1
        )
    )

    historico = historico.withColumn(
        "historico_continuo",
        F.when(
            F.col("qtd_safras") ==
            F.col("qtd_safras_esperadas"),
            1
        ).otherwise(0)
    )

    print("\nDistribuição da continuidade:")

    (
        historico
        .groupBy("historico_continuo")
        .count()
        .show()
    )

    return historico

In [11]:
analisar_continuidade_historico(dfs_members)

ANÁLISE DE CONTINUIDADE DO HISTÓRICO

Distribuição da continuidade:
+------------------+-------+
|historico_continuo|  count|
+------------------+-------+
|                 1|1110844|
|                 0| 177828|
+------------------+-------+



DataFrame[msno: string, primeira_safra: date, ultima_safra: date, qtd_safras: bigint, qtd_safras_esperadas: int, historico_continuo: int]

# Logs

engajamento atual
frequência de uso
intensidade de consumo
qualidade/completude do consumo
diversidade de consumo
tendência recente
queda ou aumento de engajamento
ausência de uso

##  Formatação

In [12]:
def preprocess_logs(df: DataFrame) -> DataFrame:
    """
    ============================================================
    ETAPA 1 - PRÉ-PROCESSAMENTO DOS LOGS
    ============================================================

    Objetivo:
    ----------
    Preparar os logs para posterior agregação por:

        cliente × safra

    Regras:
    -------
    - Não exige histórico contínuo.
    - Não define o público-alvo.
    - Não utiliza informações futuras.
    - Não interpreta ausência de log como churn.
    """

    print("=" * 70)
    print("ETAPA 1 - PRÉ-PROCESSAMENTO DOS LOGS")
    print("=" * 70)

    # ==========================================================
    # 1. REGISTROS INICIAIS
    # ==========================================================

    registros_iniciais = df.count()

    print(
        f"Registros iniciais: "
        f"{registros_iniciais:,}"
    )

    # ==========================================================
    # 2. VALIDAÇÃO DAS COLUNAS
    # ==========================================================

    colunas_obrigatorias = [
        IDENTIFY,
        TIME_COL,
        "num_25",
        "num_50",
        "num_75",
        "num_985",
        "num_100",
        "num_unq",
        "total_secs"
    ]

    colunas_faltantes = [
        c
        for c in colunas_obrigatorias
        if c not in df.columns
    ]

    if colunas_faltantes:
        raise ValueError(
            "Colunas obrigatórias ausentes nos logs: "
            f"{colunas_faltantes}"
        )

    print("✓ Todas as colunas obrigatórias estão presentes.")

    # ==========================================================
    # 3. IDENTIFICADOR
    # ==========================================================

    df = df.withColumn(
        IDENTIFY,
        F.trim(
            F.col(IDENTIFY).cast("string")
        )
    )

    # ==========================================================
    # 4. SAFRA
    # ==========================================================

    df = df.withColumn(
        TIME_COL,
        F.trim(
            F.col(TIME_COL).cast("string")
        )
    )

    # ==========================================================
    # 5. DATA DA SAFRA
    # ==========================================================

    df = df.withColumn(
        "safra_date_dt",
        F.to_date(
            F.concat(
                F.col(TIME_COL),
                F.lit("01")
            ),
            "yyyyMMdd"
        )
    )

    # ==========================================================
    # 6. VARIÁVEIS NUMÉRICAS
    # ==========================================================

    numeric_cols = [
        "num_25",
        "num_50",
        "num_75",
        "num_985",
        "num_100",
        "num_unq",
        "total_secs"
    ]

    for col_name in numeric_cols:

        df = df.withColumn(
            col_name,
            F.col(col_name).cast("double")
        )

        # Valores negativos → zero
        df = df.withColumn(
            col_name,
            F.when(
                F.col(col_name) < 0,
                F.lit(0.0)
            ).otherwise(
                F.col(col_name)
            )
        )

        # NULL → zero
        df = df.withColumn(
            col_name,
            F.coalesce(
                F.col(col_name),
                F.lit(0.0)
            )
        )

    # ==========================================================
    # 7. REMOVER CHAVES INVÁLIDAS
    # ==========================================================

    registros_antes_chave = df.count()

    df = df.dropna(
        subset=[
            IDENTIFY,
            TIME_COL,
            "safra_date_dt"
        ]
    )

    registros_depois_chave = df.count()

    print("\nChaves temporais:")
    print(
        f"  Antes : {registros_antes_chave:,}"
    )
    print(
        f"  Depois: {registros_depois_chave:,}"
    )
    print(
        f"  Removidos: "
        f"{registros_antes_chave - registros_depois_chave:,}"
    )

    # ==========================================================
    # 8. CLIENTES VÁLIDOS
    # ==========================================================

    ids_validos = (
        spark.read
        .parquet(output_analysis_path)
        .select(IDENTIFY)
        .distinct()
    )

    registros_antes_clientes = df.count()

    df = df.join(
        ids_validos,
        on=IDENTIFY,
        how="inner"
    )

    registros_depois_clientes = df.count()

    print("\nFiltro de clientes presentes nas três bases:")
    print(
        f"  Antes : {registros_antes_clientes:,}"
    )
    print(
        f"  Depois: {registros_depois_clientes:,}"
    )
    print(
        f"  Removidos: "
        f"{registros_antes_clientes - registros_depois_clientes:,}"
    )

    # ==========================================================
    # 9. RESULTADO
    # ==========================================================

    print("\nResultado do pré-processamento:")
    print(
        f"  Registros finais: "
        f"{registros_depois_clientes:,}"
    )

    return df

##  Agregação Mensal

In [13]:
def aggregate_logs_monthly(
    df: DataFrame
) -> DataFrame:

    """
    ============================================================
    ETAPA 2 - AGREGAÇÃO MENSAL DOS LOGS
    ============================================================

    Unidade final:

        1 cliente × 1 safra
    """

    print("\n" + "=" * 70)
    print("ETAPA 2 - AGREGAÇÃO MENSAL")
    print("=" * 70)

    df_month = (
        df
        .groupBy(
            IDENTIFY,
            TIME_COL,
            "safra_date_dt"
        )
        .agg(

            # Volume de consumo
            F.sum("num_25").alias("num_25"),
            F.sum("num_50").alias("num_50"),
            F.sum("num_75").alias("num_75"),
            F.sum("num_985").alias("num_985"),
            F.sum("num_100").alias("num_100"),

            # Diversidade
            F.sum("num_unq").alias("num_unq"),

            # Tempo total de consumo
            F.sum("total_secs").alias("total_secs"),

            # Quantidade de dias com consumo
            F.sum(
                F.when(
                    F.col("total_secs") > 0,
                    1
                ).otherwise(0)
            ).alias("dias_ativos")
        )
    )

    # ==========================================================
    # TOTAL DE PLAYS
    # ==========================================================

    df_month = df_month.withColumn(
        "total_plays",
        (
            F.col("num_25") +
            F.col("num_50") +
            F.col("num_75") +
            F.col("num_985") +
            F.col("num_100")
        )
    )

    # ==========================================================
    # TAXA DE COMPLETUDE
    # ==========================================================

    df_month = df_month.withColumn(
        "taxa_completude",
        F.when(
            F.col("total_plays") > 0,
            F.col("num_100") /
            F.col("total_plays")
        ).otherwise(0.0)
    )

    # ==========================================================
    # SKIP RATIO
    # ==========================================================

    df_month = df_month.withColumn(
        "skip_ratio",
        F.when(
            F.col("total_plays") > 0,
            (
                F.col("num_25") +
                F.col("num_50")
            ) / F.col("total_plays")
        ).otherwise(0.0)
    )

    # ==========================================================
    # TEMPO MÉDIO POR PLAY
    # ==========================================================

    df_month = df_month.withColumn(
        "avg_secs_per_play",
        F.when(
            F.col("total_plays") > 0,
            F.col("total_secs") /
            F.col("total_plays")
        ).otherwise(0.0)
    )

    # ==========================================================
    # DIVERSIDADE RELATIVA
    # ==========================================================

    df_month = df_month.withColumn(
        "diversidade_ratio",
        F.when(
            F.col("total_plays") > 0,
            F.col("num_unq") /
            F.col("total_plays")
        ).otherwise(0.0)
    )

    print(
        f"Cliente × safra: "
        f"{df_month.count():,}"
    )

    return df_month

##  Behavior

In [14]:
def build_behavior_features_logs(
    df: DataFrame
) -> DataFrame:

    """
    ============================================================
    ETAPA 3 - FEATURES DE COMPORTAMENTO DOS LOGS
    ============================================================

    Todas as features são calculadas utilizando apenas
    informações disponíveis até M0.

    Não existe utilização de M+1, M+2 ou M+3.
    """

    print("\n" + "=" * 70)
    print("ETAPA 3 - FEATURES DE COMPORTAMENTO")
    print("=" * 70)

    # ==========================================================
    # 1. ÍNDICE MENSAL
    # ==========================================================

    df = df.withColumn(
        "_mes_indice",
        (
            F.year("safra_date_dt") * 12
            + F.month("safra_date_dt")
        )
    )

    # ==========================================================
    # 2. JANELA DOS ÚLTIMOS 3 MESES
    # ==========================================================

    w_3m = (
        Window
        .partitionBy(IDENTIFY)
        .orderBy("_mes_indice")
        .rangeBetween(-2, 0)
    )

    # ==========================================================
    # 3. VOLUME RECENTE
    # ==========================================================

    df = df.withColumn(
        "total_plays_3m",
        F.sum("total_plays").over(w_3m)
    )

    df = df.withColumn(
        "total_secs_3m",
        F.sum("total_secs").over(w_3m)
    )

    df = df.withColumn(
        "num_unq_3m",
        F.sum("num_unq").over(w_3m)
    )

    # ==========================================================
    # 4. MESES COM ATIVIDADE
    # ==========================================================

    df = df.withColumn(
        "_mes_ativo",
        (
            F.col("total_plays") > 0
        ).cast("int")
    )

    df = df.withColumn(
        "meses_ativos_3m",
        F.sum("_mes_ativo").over(w_3m)
    )

    # ==========================================================
    # 5. MÉDIAS DOS ÚLTIMOS 3 MESES
    # ==========================================================

    df = df.withColumn(
        "media_plays_3m",
        F.col("total_plays_3m") / 3.0
    )

    df = df.withColumn(
        "media_secs_3m",
        F.col("total_secs_3m") / 3.0
    )

    df = df.withColumn(
        "media_unq_3m",
        F.col("num_unq_3m") / 3.0
    )

    # ==========================================================
    # 6. VARIAÇÃO M0 VS M-1
    # ==========================================================

    w_lag = (
        Window
        .partitionBy(IDENTIFY)
        .orderBy("_mes_indice")
    )

    for col_name in [
        "total_plays",
        "total_secs",
        "num_unq"
    ]:

        lag_col = f"_lag_{col_name}"

        df = df.withColumn(
            lag_col,
            F.lag(col_name).over(w_lag)
        )

        df = df.withColumn(
            f"var_{col_name}",
            F.when(
                F.col(lag_col) > 0,
                (
                    F.col(col_name) -
                    F.col(lag_col)
                ) / F.col(lag_col)
            ).otherwise(0.0)
        )

    # ==========================================================
    # 7. COMPARAÇÃO M0 VS MÉDIA ANTERIOR
    # ==========================================================

    w_anterior = (
        Window
        .partitionBy(IDENTIFY)
        .orderBy("_mes_indice")
        .rangeBetween(-3, -1)
    )

    df = df.withColumn(
        "_media_plays_anterior",
        F.avg("total_plays").over(w_anterior)
    )

    df = df.withColumn(
        "_media_secs_anterior",
        F.avg("total_secs").over(w_anterior)
    )

    df = df.withColumn(
        "var_plays_vs_media_3m",
        F.when(
            F.col("_media_plays_anterior") > 0,
            (
                F.col("total_plays") -
                F.col("_media_plays_anterior")
            ) / F.col("_media_plays_anterior")
        ).otherwise(0.0)
    )

    df = df.withColumn(
        "var_secs_vs_media_3m",
        F.when(
            F.col("_media_secs_anterior") > 0,
            (
                F.col("total_secs") -
                F.col("_media_secs_anterior")
            ) / F.col("_media_secs_anterior")
        ).otherwise(0.0)
    )

    # ==========================================================
    # 8. LIMPEZA
    # ==========================================================

    colunas_auxiliares = [
        c
        for c in df.columns
        if c.startswith("_")
    ]

    if colunas_auxiliares:
        df = df.drop(*colunas_auxiliares)

    return df

##  Join Público

In [15]:
def keep_logs_publico_alvo(
    df_logs_features: DataFrame,
    publico_alvo_path: str
) -> DataFrame:

    """
    ============================================================
    ETAPA 4 - JUNÇÃO COM O PÚBLICO-ALVO
    ============================================================

    O público é a referência.

    Portanto:

        público LEFT JOIN logs

    Isso preserva clientes ativos em M0 que eventualmente
    não possuem registro de uso nos logs.
    """

    print("\n" + "=" * 70)
    print("ETAPA 4 - JOIN COM PÚBLICO-ALVO")
    print("=" * 70)

    publico = (
        spark.read
        .parquet(publico_alvo_path)
        .select(
            IDENTIFY,
            "safra",
            "safra_date_dt",
            "churn_m3"
        )
        .dropDuplicates(
            [IDENTIFY, "safra"]
        )
    )

    print(
        f"Observações no público: "
        f"{publico.count():,}"
    )

    # ==========================================================
    # JOIN
    # ==========================================================

    df_final = (
        publico.alias("p")
        .join(
            df_logs_features.alias("l"),
            on=[
                F.col(f"p.{IDENTIFY}") ==
                F.col(f"l.{IDENTIFY}"),

                F.col("p.safra") ==
                F.col("l.safra")
            ],
            how="left"
        )
        .select(
            F.col(f"p.{IDENTIFY}"),
            F.col("p.safra"),
            F.col("p.safra_date_dt"),
            F.col("p.churn_m3"),

            *[
                F.col(f"l.{c}")
                for c in df_logs_features.columns
                if c not in [
                    IDENTIFY,
                    TIME_COL,
                    "safra",
                    "safra_date_dt"
                ]
            ]
        )
    )

    # ==========================================================
    # FLAG DE AUSÊNCIA DE USO
    # ==========================================================

    df_final = df_final.withColumn(
        "flag_sem_uso_m0",
        F.when(
            F.col("total_plays").isNull(),
            1
        ).otherwise(0)
    )

    return df_final

##  Logs Ausentes

In [16]:
def preencher_ausencia_logs(
    df: DataFrame
) -> DataFrame:

    """
    Clientes sem registro nos logs em M0 são mantidos.

    A ausência é marcada por flag_sem_uso_m0 e as métricas
    comportamentais são preenchidas com zero.
    """

    metricas_logs = [
        "num_25",
        "num_50",
        "num_75",
        "num_985",
        "num_100",
        "num_unq",
        "total_secs",
        "dias_ativos",
        "total_plays",
        "taxa_completude",
        "skip_ratio",
        "avg_secs_per_play",
        "diversidade_ratio",
        "total_plays_3m",
        "total_secs_3m",
        "num_unq_3m",
        "meses_ativos_3m",
        "media_plays_3m",
        "media_secs_3m",
        "media_unq_3m",
        "var_total_plays",
        "var_total_secs",
        "var_num_unq",
        "var_plays_vs_media_3m",
        "var_secs_vs_media_3m"
    ]

    for col_name in metricas_logs:

        if col_name in df.columns:

            df = df.withColumn(
                col_name,
                F.coalesce(
                    F.col(col_name),
                    F.lit(0.0)
                )
            )

    return df

##  Pipeline Logs

In [17]:
def run_pipeline_logs(
    df,
    publico_alvo_path,
    output_path,
    overwrite=True
):
    """
    ============================================================
    PIPELINE COMPLETO - FEATURES DE LOGS
    ============================================================

    Fluxo:

        Logs amostrados
            ↓
        Preparação
            ↓
        Agregação mensal
            ↓
        Features comportamentais
            ↓
        LEFT JOIN com público-alvo
            ↓
        Tratamento de ausência de logs
            ↓
        Validações
            ↓
        Salvamento
    """

    print("\n" + "=" * 70)
    print("PIPELINE DE LOGS")
    print("=" * 70)

    # --------------------------------------------------------
    # 1. PREPARAÇÃO
    # --------------------------------------------------------

    df_logs = preprocess_logs(df)

    # --------------------------------------------------------
    # 2. AGREGAÇÃO MENSAL
    # --------------------------------------------------------

    df_logs_monthly = aggregate_logs_monthly(
        df_logs
    )

    # --------------------------------------------------------
    # 3. FEATURES COMPORTAMENTAIS
    # --------------------------------------------------------

    df_logs_features = build_behavior_features_logs(
        df_logs_monthly
    )

    # --------------------------------------------------------
    # 4. JOIN COM O PÚBLICO-ALVO
    # --------------------------------------------------------

    df_logs_final = keep_logs_publico_alvo(
        df_logs_features,
        publico_alvo_path
    )

    # --------------------------------------------------------
    # 5. TRATAMENTO DE CLIENTES SEM LOG
    # --------------------------------------------------------

    df_logs_final = preencher_ausencia_logs(
        df_logs_final
    )

    # --------------------------------------------------------
    # 6. VALIDAÇÕES
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("VALIDAÇÃO FINAL")
    print("=" * 70)

    qtd_final = df_logs_final.count()

    qtd_clientes = (
        df_logs_final
        .select(IDENTIFY)
        .distinct()
        .count()
    )

    duplicidades = (
        df_logs_final
        .groupBy(
            IDENTIFY,
            "safra"
        )
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(f"Registros finais       : {qtd_final:,}")
    print(f"Clientes distintos     : {qtd_clientes:,}")
    print(f"Duplicidades cliente+safra: {duplicidades:,}")

    print("\nAusência de logs no M0:")

    (
        df_logs_final
        .groupBy("flag_sem_uso_m0")
        .count()
        .orderBy("flag_sem_uso_m0")
        .show()
    )

    print("\nNulos finais:")

    (
        df_logs_final
        .select([
            F.sum(
                F.col(c).isNull().cast("int")
            ).alias(c)
            for c in df_logs_final.columns
        ])
        .show(truncate=False)
    )

    # --------------------------------------------------------
    # 7. SALVAR
    # --------------------------------------------------------

    (
        df_logs_final
        .write
        .mode("overwrite" if overwrite else "error")
        .parquet(output_path)
    )

    print("\n" + "=" * 70)
    print("PIPELINE FINALIZADO")
    print("=" * 70)

    print(f"Arquivo salvo em:")
    print(output_path)

    return df_logs_final

In [18]:
%%time
# ============================================================
# PIPELINE - FEATURES DE LOGS
# ============================================================

input_logs = (
    "/content/drive/MyDrive/SANTANDER/"
    "user_logs.parquet"
)

publico_alvo_path = (
    "/content/drive/MyDrive/SANTANDER/"
    "features_members.parquet"
)

output_logs = (
    "/content/drive/MyDrive/SANTANDER/"
    "features_logs.parquet"
)


# ============================================================
# 1. LEITURA DOS LOGS
# ============================================================

print("=" * 70)
print("LEITURA DOS LOGS")
print("=" * 70)

df_logs = spark.read.parquet(input_logs)

print(f"Registros originais : {df_logs.count():,}")
print(f"Colunas              : {len(df_logs.columns)}")

'''
# ============================================================
# 2. AMOSTRA ESTRATIFICADA POR SAFRA
# ============================================================

SEED = 42
SAMPLE_FRACTION = 0.10

fractions = {
    row["safra"]: SAMPLE_FRACTION
    for row in df_logs
        .select("safra")
        .distinct()
        .collect()
}

df_logs = df_logs.sampleBy(
    col="safra",
    fractions=fractions,
    seed=SEED
)

print("\n" + "=" * 70)
print("AMOSTRA ESTRATIFICADA")
print("=" * 70)

print(f"Fração utilizada : {SAMPLE_FRACTIO.0%}")
print(f"Registros amostra: {df_logs.count():,}")
'''

print("\nDistribuição por safra:")
(
    df_logs
    .groupBy("safra")
    .count()
    .orderBy("safra")
    .show(50, truncate=False)
)


# ============================================================
# 3. EXECUÇÃO DO PIPELINE DE LOGS
# ============================================================

df_logs_final = run_pipeline_logs(
    df=df_logs,
    publico_alvo_path=publico_alvo_path,
    output_path=output_logs,
    overwrite=True
)

LEITURA DOS LOGS
Registros originais : 26,758,971
Colunas              : 9

Distribuição por safra:
+------+-------+
|safra |count  |
+------+-------+
|201501|937789 |
|201502|933040 |
|201503|944739 |
|201504|939930 |
|201505|924216 |
|201506|916862 |
|201507|871491 |
|201508|920129 |
|201509|903194 |
|201510|1012953|
|201511|1041975|
|201512|1039271|
|201601|1076712|
|201602|1041248|
|201603|1048941|
|201604|1042406|
|201605|1056491|
|201606|1081181|
|201607|1102807|
|201608|1103078|
|201609|1112601|
|201610|1139089|
|201611|1183088|
|201612|1135573|
|201701|1136003|
|201702|1114164|
+------+-------+


PIPELINE DE LOGS
ETAPA 1 - PRÉ-PROCESSAMENTO DOS LOGS
Registros iniciais: 26,758,971
✓ Todas as colunas obrigatórias estão presentes.

Chaves temporais:
  Antes : 26,758,971
  Depois: 26,758,971
  Removidos: 0

Filtro de clientes presentes nas três bases:
  Antes : 26,758,971
  Depois: 14,841,572
  Removidos: 11,917,399

Resultado do pré-processamento:
  Registros finais: 14,841,572

E

# Transactions

##  Formatação

In [19]:
def preprocess_transactions(df: DataFrame) -> DataFrame:
    """
    ============================================================
    ETAPA 1 - PREPARAÇÃO DAS TRANSACTIONS
    ============================================================

    Objetivo:
    ----------
    1. Padronizar tipos.
    2. Validar o identificador do cliente.
    3. Converter datas.
    4. Criar a safra da transação.
    5. Padronizar variáveis numéricas e flags.
    6. Preservar somente informações disponíveis até a data
       da própria transação.

    IMPORTANTE:
    -----------
    Esta etapa NÃO define o público-alvo.

    O público será definido posteriormente pelo dataset
    de Members.

    Também não exigimos:
        - histórico mínimo;
        - transação em todos os meses;
        - transação no M0.
    """

    print("=" * 70)
    print("ETAPA 1 - PREPARAÇÃO DAS TRANSACTIONS")
    print("=" * 70)

    # ----------------------------------------------------------
    # 1. Identificador
    # ----------------------------------------------------------

    df = (
        df
        .withColumn(
            IDENTIFY,
            F.trim(F.col(IDENTIFY).cast("string"))
        )
        .filter(
            F.col(IDENTIFY).isNotNull() &
            (F.col(IDENTIFY) != "")
        )
    )

    # ----------------------------------------------------------
    # 2. Data da transação
    # ----------------------------------------------------------

    df = df.withColumn(
        "transaction_date",
        F.coalesce(
            F.to_date(
                F.col("transaction_date").cast("string"),
                "yyyyMMdd"
            ),
            F.to_date(
                F.col("transaction_date").cast("string"),
                "yyyy-MM-dd"
            )
        )
    )

    # ----------------------------------------------------------
    # 3. Safra da transação
    # ----------------------------------------------------------

    df = df.withColumn(
        "safra",
        F.date_format(
            F.col("transaction_date"),
            "yyyyMM"
        )
    )

    df = df.withColumn(
        "safra_date_dt",
        F.to_date(
            F.concat(
                F.col("safra"),
                F.lit("01")
            ),
            "yyyyMMdd"
        )
    )

    # ----------------------------------------------------------
    # 4. Data de expiração
    # ----------------------------------------------------------

    df = df.withColumn(
        "membership_expire_date",
        F.coalesce(
            F.to_date(
                F.col("membership_expire_date").cast("string"),
                "yyyyMMdd"
            ),
            F.to_date(
                F.col("membership_expire_date").cast("string"),
                "yyyy-MM-dd"
            )
        )
    )

    # ----------------------------------------------------------
    # 5. Tipos numéricos
    # ----------------------------------------------------------

    df = (
        df
        .withColumn(
            "payment_method_id",
            F.col("payment_method_id").cast("integer")
        )
        .withColumn(
            "payment_plan_days",
            F.col("payment_plan_days").cast("integer")
        )
        .withColumn(
            "plan_list_price",
            F.col("plan_list_price").cast("double")
        )
        .withColumn(
            "actual_amount_paid",
            F.col("actual_amount_paid").cast("double")
        )
        .withColumn(
            "is_auto_renew",
            F.col("is_auto_renew").cast("integer")
        )
        .withColumn(
            "is_cancel",
            F.col("is_cancel").cast("integer")
        )
    )

    # ----------------------------------------------------------
    # 6. Valores financeiros inválidos
    # ----------------------------------------------------------

    df = (
        df
        .withColumn(
            "plan_list_price",
            F.when(
                F.col("plan_list_price") < 0,
                None
            ).otherwise(
                F.col("plan_list_price")
            )
        )
        .withColumn(
            "actual_amount_paid",
            F.when(
                F.col("actual_amount_paid") < 0,
                None
            ).otherwise(
                F.col("actual_amount_paid")
            )
        )
    )

    # ----------------------------------------------------------
    # 7. Padronização das flags
    # ----------------------------------------------------------

    df = (
        df
        .withColumn(
            "is_auto_renew",
            F.when(F.col("is_auto_renew") == 1, 1)
             .when(F.col("is_auto_renew") == 0, 0)
             .otherwise(None)
        )
        .withColumn(
            "is_cancel",
            F.when(F.col("is_cancel") == 1, 1)
             .when(F.col("is_cancel") == 0, 0)
             .otherwise(None)
        )
    )

    # ----------------------------------------------------------
    # 8. Validações assertivas
    # ----------------------------------------------------------

    assert (
        df.filter(F.col(IDENTIFY).isNull()).count() == 0
    ), "Existem identificadores nulos."

    assert (
        df.filter(F.col("transaction_date").isNull()).count() == 0
    ), "Existem transaction_date inválidas."

    assert (
        df.filter(
            F.col("plan_list_price") < 0
        ).count() == 0
    ), "Existem plan_list_price negativos."

    assert (
        df.filter(
            F.col("actual_amount_paid") < 0
        ).count() == 0
    ), "Existem actual_amount_paid negativos."

    print(f"Registros: {df.count():,}")
    print(f"Clientes distintos: {df.select(IDENTIFY).distinct().count():,}")

    print("\nPeríodo:")
    df.select(
        F.min("transaction_date").alias("min_date"),
        F.max("transaction_date").alias("max_date")
    ).show()

    return df

##  Agregação Mensal

In [20]:
def aggregate_transactions_monthly(
    df: DataFrame
) -> DataFrame:
    """
    ============================================================
    ETAPA 2 - AGREGAÇÃO MENSAL
    ============================================================

    Unidade:
        cliente × safra

    Objetivo:
        Transformar as transações do mês em uma única
        observação por cliente.

    As variáveis representam o comportamento financeiro/
    transacional observado naquele mês.
    """

    print("=" * 70)
    print("ETAPA 2 - AGREGAÇÃO MENSAL")
    print("=" * 70)

    df_monthly = (
        df
        .groupBy(
            IDENTIFY,
            "safra",
            "safra_date_dt"
        )
        .agg(

            # --------------------------------------------------
            # Receita
            # --------------------------------------------------

            F.sum(
                F.coalesce(
                    F.col("actual_amount_paid"),
                    F.lit(0.0)
                )
            ).alias("receita_mes"),

            F.avg(
                "actual_amount_paid"
            ).alias("ticket_medio"),

            # --------------------------------------------------
            # Quantidade
            # --------------------------------------------------

            F.count("*").alias(
                "qtd_transacoes"
            ),

            # --------------------------------------------------
            # Cancelamentos
            # --------------------------------------------------

            F.sum(
                F.coalesce(
                    F.col("is_cancel"),
                    F.lit(0)
                )
            ).alias(
                "cancelamentos_mes"
            ),

            # --------------------------------------------------
            # Estado do plano no mês
            #
            # max_by pega o valor associado à última
            # transaction_date do mês.
            # --------------------------------------------------

            F.max_by(
                "payment_method_id",
                "transaction_date"
            ).alias(
                "payment_method_id"
            ),

            F.max_by(
                "payment_plan_days",
                "transaction_date"
            ).alias(
                "payment_plan_days"
            ),

            F.max_by(
                "plan_list_price",
                "transaction_date"
            ).alias(
                "plan_list_price"
            ),

            F.max_by(
                "actual_amount_paid",
                "transaction_date"
            ).alias(
                "actual_amount_paid"
            ),

            F.max_by(
                "is_auto_renew",
                "transaction_date"
            ).alias(
                "is_auto_renew"
            ),

            F.max_by(
                "membership_expire_date",
                "transaction_date"
            ).alias(
                "membership_expire_date"
            ),

            F.max_by(
                "is_cancel",
                "transaction_date"
            ).alias(
                "is_cancel"
            )
        )
    )

    # ----------------------------------------------------------
    # Validação fundamental:
    # uma linha por cliente × safra
    # ----------------------------------------------------------

    duplicados = (
        df_monthly
        .groupBy(IDENTIFY, "safra")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicados == 0, (
        "Existem múltiplas linhas para cliente × safra."
    )

    print(
        f"Registros cliente × safra: "
        f"{df_monthly.count():,}"
    )

    return df_monthly

##  Features

In [21]:
def build_transaction_features(
    df: DataFrame
) -> DataFrame:
    """
    ============================================================
    ETAPA 3 - FEATURES DE TRANSACTIONS
    ============================================================

    Objetivo:
        Criar variáveis que descrevem o comportamento
        transacional do cliente até a safra atual.

    REGRA:
        Nenhuma feature pode utilizar informação posterior
        à safra atual.

    Portanto:
        lag  -> permitido
        lead -> proibido

    Observação:
        As transações são naturalmente esparsas.
        Por isso NÃO utilizamos rowsBetween para interpretar
        "últimos 3 meses", pois isso poderia confundir
        meses de calendário com observações existentes.
    """

    print("=" * 70)
    print("ETAPA 3 - FEATURES DE TRANSACTIONS")
    print("=" * 70)

    window_cliente = (
        Window
        .partitionBy(IDENTIFY)
        .orderBy("safra_date_dt")
    )

    # ----------------------------------------------------------
    # Histórico imediato
    # ----------------------------------------------------------

    df = (
        df
        .withColumn(
            "receita_anterior",
            F.lag("receita_mes").over(
                window_cliente
            )
        )
        .withColumn(
            "preco_anterior",
            F.lag("plan_list_price").over(
                window_cliente
            )
        )
        .withColumn(
            "plano_anterior",
            F.lag("payment_plan_days").over(
                window_cliente
            )
        )
        .withColumn(
            "data_transacao_anterior",
            F.lag("safra_date_dt").over(
                window_cliente
            )
        )
        .withColumn(
            "auto_renew_anterior",
            F.lag("is_auto_renew").over(
                window_cliente
            )
        )
    )

    # ----------------------------------------------------------
    # Existência de observação anterior
    # ----------------------------------------------------------

    df = df.withColumn(
        "tem_observacao_anterior",
        F.when(
            F.col("data_transacao_anterior").isNotNull(),
            1
        ).otherwise(0)
    )

    # ----------------------------------------------------------
    # Mudança de receita
    # ----------------------------------------------------------

    df = df.withColumn(
        "delta_receita",
        F.when(
            F.col("receita_anterior").isNotNull(),
            F.col("receita_mes") -
            F.col("receita_anterior")
        )
    )

    df = df.withColumn(
        "var_receita",
        F.when(
            F.col("receita_anterior").isNull(),
            None
        )
        .when(
            F.col("receita_anterior") == 0,
            0.0
        )
        .otherwise(
            (
                F.col("receita_mes") -
                F.col("receita_anterior")
            )
            /
            F.col("receita_anterior")
        )
    )

    # ----------------------------------------------------------
    # Mudança de preço
    # ----------------------------------------------------------

    df = df.withColumn(
        "mudanca_preco",
        F.when(
            F.col("preco_anterior").isNotNull() &
            F.col("plan_list_price").isNotNull(),
            F.col("plan_list_price") -
            F.col("preco_anterior")
        )
    )

    # ----------------------------------------------------------
    # Mudança de plano
    # ----------------------------------------------------------

    df = df.withColumn(
        "upgrade_plano",
        F.when(
            F.col("plano_anterior").isNotNull() &
            F.col("payment_plan_days").isNotNull() &
            (
                F.col("payment_plan_days") >
                F.col("plano_anterior")
            ),
            1
        ).otherwise(0)
    )

    df = df.withColumn(
        "downgrade_plano",
        F.when(
            F.col("plano_anterior").isNotNull() &
            F.col("payment_plan_days").isNotNull() &
            (
                F.col("payment_plan_days") <
                F.col("plano_anterior")
            ),
            1
        ).otherwise(0)
    )

    # ----------------------------------------------------------
    # Intervalo entre transações
    # ----------------------------------------------------------

    df = df.withColumn(
        "intervalo_transacoes_meses",
        F.when(
            F.col("data_transacao_anterior").isNotNull(),
            (
                F.year("safra_date_dt") * 12
                + F.month("safra_date_dt")
                -
                (
                    F.year(
                        "data_transacao_anterior"
                    ) * 12
                    +
                    F.month(
                        "data_transacao_anterior"
                    )
                )
            ).cast("int")
        )
    )

    df = df.withColumn(
        "flag_gap_transacoes",
        F.when(
            F.col("intervalo_transacoes_meses") > 1,
            1
        ).otherwise(0)
    )

    # ----------------------------------------------------------
    # Histórico de auto-renovação
    # ----------------------------------------------------------

    df = df.withColumn(
        "taxa_auto_renew_hist",
        F.avg(
            F.col("is_auto_renew")
        ).over(
            window_cliente.rowsBetween(
                Window.unboundedPreceding,
                -1
            )
        )
    )

    # ----------------------------------------------------------
    # Remove colunas auxiliares
    # ----------------------------------------------------------

    df = df.drop(
        "receita_anterior",
        "preco_anterior",
        "plano_anterior",
        "data_transacao_anterior",
        "auto_renew_anterior"
    )

    # ----------------------------------------------------------
    # Validação: nenhuma feature usa futuro
    #
    # Todas as variáveis históricas são baseadas em LAG.
    # ----------------------------------------------------------

    print(
        f"Registros com features: "
        f"{df.count():,}"
    )

    return df

##  Join Público

In [22]:
def join_transactions_publico(
    df_transactions: DataFrame,
    publico_alvo_path: str
) -> DataFrame:
    """
    ============================================================
    ETAPA 4 - JOIN COM O PÚBLICO-ALVO
    ============================================================

    Regra:
        O público-alvo é a referência.

    Portanto:
        PUBLICO LEFT JOIN TRANSACTIONS

    Isso preserva clientes ativos no M0 mesmo sem
    transação registrada no mês.
    """

    print("=" * 70)
    print("ETAPA 4 - JOIN COM O PÚBLICO-ALVO")
    print("=" * 70)

    df_publico = (
        spark.read
        .parquet(publico_alvo_path)
        .select(
            IDENTIFY,
            "safra"
        )
        .dropDuplicates(
            [IDENTIFY, "safra"]
        )
    )

    df_publico = df_publico.withColumn(
        "safra",
        F.col("safra").cast("string")
    )

    # ----------------------------------------------------------
    # Quantidade de observações do público
    # ----------------------------------------------------------

    qtd_publico = df_publico.count()

    # ----------------------------------------------------------
    # LEFT JOIN
    # ----------------------------------------------------------

    df_final = (
        df_publico
        .join(
            df_transactions,
            on=[IDENTIFY, "safra"],
            how="left"
        )
    )

    # ----------------------------------------------------------
    # Flag explícita de ausência de transação no M0
    # ----------------------------------------------------------

    df_final = df_final.withColumn(
        "flag_sem_transacao_m0",
        F.when(
            F.col("receita_mes").isNull(),
            1
        ).otherwise(0)
    )

    # ----------------------------------------------------------
    # Validação fundamental:
    # LEFT JOIN NÃO PODE ALTERAR O PÚBLICO
    # ----------------------------------------------------------

    qtd_final = df_final.count()

    assert qtd_final == qtd_publico, (
        f"LEFT JOIN alterou o número de observações do público: "
        f"{qtd_publico:,} → {qtd_final:,}"
    )

    duplicados = (
        df_final
        .groupBy(IDENTIFY, "safra")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicados == 0, (
        "O resultado final possui duplicidade em cliente × safra."
    )

    print(
        f"Público antes do JOIN : {qtd_publico:,}"
    )

    print(
        f"Público depois do JOI {qtd_final:,}"
    )

    print("\nAusência de transação no M0:")

    (
        df_final
        .groupBy("flag_sem_transacao_m0")
        .count()
        .orderBy("flag_sem_transacao_m0")
        .show()
    )

    return df_final

##  Transactiosn Null

In [23]:
def preencher_ausencia_transactions(
    df: DataFrame
) -> DataFrame:
    """
    ============================================================
    ETAPA 5 - TRATAMENTO DA AUSÊNCIA DE TRANSAÇÃO
    ============================================================

    Quando não existe transação no M0:

    Métricas de atividade:
        -> 0

    Variáveis de estado/plano:
        -> permanecem nulas

    A ausência é identificada explicitamente por:
        flag_sem_transacao_m0
    """

    print("=" * 70)
    print("ETAPA 5 - TRATAMENTO DA AUSÊNCIA DE TRANSAÇÃO")
    print("=" * 70)

    # ----------------------------------------------------------
    # Métricas em que ausência de transação = ausência de
    # atividade financeira no mês
    # ----------------------------------------------------------

    colunas_zero = [
        "receita_mes",
        "qtd_transacoes",
        "cancelamentos_mes"
    ]

    for coluna in colunas_zero:

        df = df.withColumn(
            coluna,
            F.coalesce(
                F.col(coluna),
                F.lit(0)
            )
        )

    return df

## Validations

In [24]:
def validar_transactions_final(
    df: DataFrame
) -> None:
    """
    ============================================================
    VALIDAÇÕES FINAIS - TRANSACTIONS
    ============================================================

    Validações de integridade do dataset final.
    """

    print("=" * 70)
    print("VALIDAÇÕES FINAIS - TRANSACTIONS")
    print("=" * 70)

    # ----------------------------------------------------------
    # 1. Chave cliente × safra
    # ----------------------------------------------------------

    duplicados = (
        df
        .groupBy(
            IDENTIFY,
            "safra"
        )
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    assert duplicados == 0, (
        "Existem duplicidades em cliente × safra."
    )

    print("✓ Chave cliente × safra: OK")

    # ----------------------------------------------------------
    # 2. Receita não pode ser negativa
    # ----------------------------------------------------------

    receita_negativa = (
        df
        .filter(
            F.col("receita_mes") < 0
        )
        .count()
    )

    assert receita_negativa == 0, (
        "Existem receitas mensais negativas."
    )

    print("✓ Receita mensal não negativa: OK")

    # ----------------------------------------------------------
    # 3. Quantidade de transações não negativa
    # ----------------------------------------------------------

    qtd_negativa = (
        df
        .filter(
            F.col("qtd_transacoes") < 0
        )
        .count()
    )

    assert qtd_negativa == 0, (
        "Existem quantidades de transações negativas."
    )

    print("✓ Quantidade de transações válida: OK")

    # ----------------------------------------------------------
    # 4. Flags binárias
    # ----------------------------------------------------------

    flags = [
        "flag_sem_transacao_m0",
        "upgrade_plano",
        "downgrade_plano",
        "flag_gap_transacoes"
    ]

    for coluna in flags:

        valores_invalidos = (
            df
            .filter(
                ~F.col(coluna).isin(0, 1)
            )
            .count()
        )

        assert valores_invalidos == 0, (
            f"A flag {coluna} possui valores diferentes de 0/1."
        )

    print("✓ Flags binárias: OK")

    # ----------------------------------------------------------
    # 5. Ausência de transação deve estar identificada
    # ----------------------------------------------------------

    inconsistencias = (
        df
        .filter(
            (
                (F.col("flag_sem_transacao_m0") == 1) &
                (F.col("receita_mes") != 0)
            )
            |
            (
                (F.col("flag_sem_transacao_m0") == 0) &
                F.col("receita_mes").isNull()
            )
        )
        .count()
    )

    assert inconsistencias == 0, (
        "Inconsistência entre ausência de transação e receita."
    )

    print(
        "✓ Flag de ausência de transação: OK"
    )

    print("\nTodas as validações foram aprovadas.")

##  Pipeline Transactions

In [25]:
def run_pipeline_transactions(
    df: DataFrame,
    publico_alvo_path: str,
    output_path: str,
    overwrite: bool = True
) -> DataFrame:
    """
    ============================================================
    PIPELINE COMPLETA - TRANSACTIONS
    ============================================================

    Fluxo:

        Transactions
             ↓
        Preparação
             ↓
        Agregação mensal
             ↓
        Features
             ↓
        LEFT JOIN Público
             ↓
        Tratamento da ausência
             ↓
        Validações
             ↓
        Parquet final
    """

    print("\n")
    print("#" * 70)
    print("PIPELINE DE TRANSACTIONS")
    print("#" * 70)

    # ==========================================================
    # 1. Preparação
    # ==========================================================

    df_transactions = preprocess_transactions(df)

    # ==========================================================
    # 2. Agregação cliente × safra
    # ==========================================================

    df_transactions_monthly = (
        aggregate_transactions_monthly(
            df_transactions
        )
    )

    # ==========================================================
    # 3. Features transacionais
    # ==========================================================

    df_transactions_features = (
        build_transaction_features(
            df_transactions_monthly
        )
    )

    # ==========================================================
    # 4. Join com público
    # ==========================================================

    df_transactions_final = (
        join_transactions_publico(
            df_transactions_features,
            publico_alvo_path
        )
    )

    # ==========================================================
    # 5. Tratamento de ausência
    # ==========================================================

    df_transactions_final = (
        preencher_ausencia_transactions(
            df_transactions_final
        )
    )

    # ==========================================================
    # 6. Validações
    # ==========================================================

    validar_transactions_final(
        df_transactions_final
    )

    # ==========================================================
    # 7. Salvamento
    # ==========================================================

    print("\n" + "=" * 70)
    print("SALVANDO DATASET")
    print("=" * 70)

    (
        df_transactions_final
        .write
        .mode(
            "overwrite" if overwrite else "error"
        )
        .parquet(output_path)
    )

    print(f"Arquivo salvo em:")
    print(output_path)

    print("\n" + "#" * 70)
    print("PIPELINE FINALIZADA")
    print("#" * 70)

    return df_transactions_final

In [26]:
%%time

# ============================================================
# EXECUÇÃO - TRANSACTIONS
# ============================================================

input_transactions = (
    "/content/drive/MyDrive/SANTANDER/transactions.parquet"
)

publico_alvo_path = (
    "/content/drive/MyDrive/SANTANDER/features_members.parquet"
)

output_transactions = (
    "/content/drive/MyDrive/SANTANDER/features_transactions.parquet"
)

SEED = 42
SAMPLE_FRACTION = 0.10


# ============================================================
# 1. Leitura
# ============================================================

df_transactions = spark.read.parquet(
    input_transactions
)

print("=" * 70)
print("DATASET ORIGINAL")
print("=" * 70)

print(
    f"Registros: {df_transactions.count():,}"
)

'''
# ============================================================
# 2. Amostra estratificada por safra
# ============================================================

fractions = {
    row["safra"]: SAMPLE_FRACTION
    for row in (
        df_transactions
        .select("safra")
        .distinct()
        .collect()
    )
}

df_transactions = (
    df_transactions
    .sampleBy(
        col="safra",
        fractions=fractions,
        seed=SEED
    )
)
'''

# ============================================================
# 3. Validação da amostra
# ============================================================

print("\n" + "=" * 70)
print("AMOSTRA ESTRATIFICADA")
print("=" * 70)

qtd_amostra = df_transactions.count()

print(
    f"Registros amostrados: {qtd_amostra:,}"
)

print("\nDistribuição por safra:")

(
    df_transactions
    .groupBy("safra")
    .count()
    .orderBy("safra")
    .show(50)
)


# ============================================================
# 4. Executa a pipeline
# ============================================================

df_transactions_final = run_pipeline_transactions(
    df=df_transactions,
    publico_alvo_path=publico_alvo_path,
    output_path=output_transactions,
    overwrite=True
)

DATASET ORIGINAL
Registros: 20,712,225

AMOSTRA ESTRATIFICADA
Registros amostrados: 20,712,225

Distribuição por safra:
+------+-------+
| safra|  count|
+------+-------+
|201501| 548792|
|201502| 545303|
|201503| 626488|
|201504| 564582|
|201505| 571552|
|201506| 775737|
|201507| 665280|
|201508| 705975|
|201509| 714610|
|201510| 680465|
|201511| 820345|
|201512| 861107|
|201601| 856716|
|201602| 792300|
|201603| 775469|
|201604| 774169|
|201605| 783956|
|201606| 804729|
|201607| 924032|
|201608| 966450|
|201609| 982640|
|201610|1033898|
|201611|1094941|
|201612| 968547|
|201701| 988576|
|201702| 885566|
+------+-------+



######################################################################
PIPELINE DE TRANSACTIONS
######################################################################
ETAPA 1 - PREPARAÇÃO DAS TRANSACTIONS
Registros: 20,712,225
Clientes distintos: 2,363,626

Período:
+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2015-01-01|2017-02-28|
+---

# Master

In [10]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from collections import Counter


# ============================================================
# CONFIGURAÇÃO
# ============================================================

IDENTIFY = "msno"


# ============================================================
# VALIDAÇÃO DE CHAVE
# ============================================================

def validar_chave_unica(df: DataFrame, nome_df: str):

    duplicados = (
        df
        .groupBy(IDENTIFY, "safra")
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    assert duplicados == 0, (
        f"ERRO: {nome_df} possui duplicidade em "
        f"{IDENTIFY} + safra."
    )

    print(f"✓ Chave única validada: {nome_df}")


# ============================================================
# VALIDAÇÃO DE COLUNAS DUPLICADAS
# ============================================================

def validar_colunas_duplicadas(df: DataFrame, nome_df: str):

    duplicadas = [
        coluna
        for coluna, quantidade in Counter(df.columns).items()
        if quantidade > 1
    ]

    assert not duplicadas, (
        f"ERRO: {nome_df} possui colunas duplicadas: "
        f"{duplicadas}"
    )

    print(f"✓ Nenhuma coluna duplicada: {nome_df}")


# ============================================================
# CONSTRUÇÃO DA MASTER
# ============================================================

def construir_master(
    members_path,
    logs_path,
    transactions_path
):

    print("=" * 70)
    print("CONSTRUÇÃO DA MASTER")
    print("=" * 70)

    # ========================================================
    # 1. LEITURA
    # ========================================================

    print("\n[1] Lendo datasets...")

    df_members = spark.read.parquet(members_path)
    df_logs = spark.read.parquet(logs_path)
    df_transactions = spark.read.parquet(transactions_path)

    print(f"Members      : {df_members.count():,}")
    print(f"Logs         : {df_logs.count():,}")
    print(f"Transactions : {df_transactions.count():,}")

    # ========================================================
    # 2. VALIDAR CHAVE
    # ========================================================

    print("\n[2] Validando chave cliente × safra...")

    validar_chave_unica(df_members, "MEMBERS")
    validar_chave_unica(df_logs, "LOGS")
    validar_chave_unica(df_transactions, "TRANSACTIONS")

    # ========================================================
    # 3. DEFINIR COLUNAS DE CADA DATASET
    # ========================================================
    #
    # Aqui NÃO usamos drop.
    #
    # Selecionamos explicitamente o que pode entrar na Master.
    #
    # Isso evita que:
    #   - safra_date_dt
    #   - churn_m3
    #   - is_ativo_m3
    #
    # sejam trazidos novamente de Logs/Transactions.
    #
    # ========================================================

    print("\n[3] Organizando colunas...")

    # --------------------------------------------------------
    # MEMBERS
    # --------------------------------------------------------
    #
    # Members é a fonte oficial de:
    #   - público
    #   - atividade M0
    #   - target
    #
    # Tudo que for data técnica fica fora da Master.
    # --------------------------------------------------------

    colunas_members = [
        c
        for c in df_members.columns
        if c not in [
            "safra_date_dt",
            "safra_date_m3",
            "is_ativo_m3"
        ]
    ]

    df_members = df_members.select(
        *colunas_members
    )

    # --------------------------------------------------------
    # LOGS
    # --------------------------------------------------------
    #
    # Logs entra apenas com comportamento.
    #
    # Removemos qualquer coluna que pertença à Members
    # ou ao target.
    # --------------------------------------------------------

    colunas_logs = [
        c
        for c in df_logs.columns
        if c not in [
            "churn_m3",
            "is_ativo",
            "is_ativo_m0",
            "is_ativo_m3",
            "safra_date_dt",
            "safra_date_m3"
        ]
    ]

    df_logs = df_logs.select(
        *colunas_logs
    )

    # --------------------------------------------------------
    # TRANSACTIONS
    # --------------------------------------------------------

    colunas_transactions = [
        c
        for c in df_transactions.columns
        if c not in [
            "churn_m3",
            "is_ativo",
            "is_ativo_m0",
            "is_ativo_m3",
            "safra_date_dt",
            "safra_date_m3"
        ]
    ]

    df_transactions = df_transactions.select(
        *colunas_transactions
    )

    # ========================================================
    # 4. VERIFICAÇÃO ANTES DOS JOINS
    # ========================================================

    print("\nColunas técnicas antes dos joins:")

    for nome, df in [
        ("MEMBERS", df_members),
        ("LOGS", df_logs),
        ("TRANSACTIONS", df_transactions)
    ]:

        tecnicas = [
            c
            for c in [
                "safra_date_dt",
                "safra_date_m3",
                "is_ativo_m3",
                "churn_m3"
            ]
            if c in df.columns
        ]

        print(f"{nome}: {tecnicas}")

    # O esperado é:
    #
    # MEMBERS:
    #   ['churn_m3']
    #
    # LOGS:
    #   []
    #
    # TRANSACTIONS:
    #   []

    # ========================================================
    # 5. CRIAR MASTER
    # ========================================================

    print("\n[4] Criando Master a partir de Members...")

    df_master = df_members

    qtd_publico = df_master.count()

    print(
        f"Quantidade de observações do público: "
        f"{qtd_publico:,}"
    )

    # ========================================================
    # 6. JOIN LOGS
    # ========================================================

    print("\n[5] Adicionando comportamento de uso...")

    df_master = (
        df_master
        .join(
            df_logs,
            on=[IDENTIFY, "safra"],
            how="left"
        )
    )

    qtd_pos_logs = df_master.count()

    assert qtd_pos_logs == qtd_publico, (
        "ERRO: JOIN com Logs alterou "
        "a quantidade de observações."
    )

    print(
        f"✓ Após Logs: {qtd_pos_logs:,} observações"
    )

    # ========================================================
    # 7. FLAG DE AUSÊNCIA DE USO
    # ========================================================

    df_master = df_master.withColumn(
        "flag_sem_uso_m0",
        F.when(
            F.col("total_plays").isNull(),
            1
        ).otherwise(0)
    )

    # ========================================================
    # 8. JOIN TRANSACTIONS
    # ========================================================

    print("\n[6] Adicionando informações financeiras...")

    df_master = (
        df_master
        .join(
            df_transactions,
            on=[IDENTIFY, "safra"],
            how="left"
        )
    )

    qtd_pos_transactions = df_master.count()

    assert qtd_pos_transactions == qtd_publico, (
        "ERRO: JOIN com Transactions alterou "
        "a quantidade de observações."
    )

    print(
        f"✓ Após Transactions: "
        f"{qtd_pos_transactions:,} observações"
    )

    # ========================================================
    # 9. FLAG DE AUSÊNCIA DE TRANSAÇÃO
    # ========================================================
    #
    # ATENÇÃO:
    # Isso ocorre antes de preencher receita_mes com zero.
    # ========================================================

    df_master = df_master.withColumn(
        "flag_sem_transacao_m0",
        F.when(
            F.col("receita_mes").isNull(),
            1
        ).otherwise(0)
    )

    # ========================================================
    # 10. RENTABILIDADE
    # ========================================================

    print("\n[7] Criando métricas de rentabilidade...")

    df_master = df_master.withColumn(
        "custo_servico_m0",
        F.lit(50)
        + F.lit(0.0051) * F.coalesce(
            F.col("num_unq"),
            F.lit(0)
        )
        + F.lit(0.0001) * F.coalesce(
            F.col("total_secs"),
            F.lit(0)
        )
    )

    df_master = df_master.withColumn(
        "margem_m0",
        F.coalesce(
            F.col("receita_mes"),
            F.lit(0)
        )
        - F.col("custo_servico_m0")
    )

    df_master = df_master.withColumn(
        "margem_percentual_m0",
        F.when(
            F.col("receita_mes") > 0,
            F.col("margem_m0") /
            F.col("receita_mes")
        )
    )

    df_master = df_master.withColumn(
        "receita_por_hora_m0",
        F.when(
            F.col("total_secs") > 0,
            F.col("receita_mes") /
            (F.col("total_secs") / F.lit(3600))
        )
    )

    # ========================================================
    # 11. PREENCHER MÉTRICAS DE USO
    # ========================================================

    colunas_uso_zero = [
        "total_plays",
        "total_secs",
        "num_unq",
        "dias_ativos",
        "total_plays_3m",
        "total_secs_3m",
        "num_unq_3m",
        "meses_ativos_3m",
        "media_plays_3m",
        "media_secs_3m",
        "media_unq_3m"
    ]

    for coluna in colunas_uso_zero:

        if coluna in df_master.columns:

            df_master = df_master.withColumn(
                coluna,
                F.coalesce(
                    F.col(coluna),
                    F.lit(0)
                )
            )

    # ========================================================
    # 12. PREENCHER MÉTRICAS DE TRANSAÇÃO
    # ========================================================

    colunas_transacao_zero = [
        "receita_mes",
        "qtd_transacoes",
        "cancelamentos_mes"
    ]

    for coluna in colunas_transacao_zero:

        if coluna in df_master.columns:

            df_master = df_master.withColumn(
                coluna,
                F.coalesce(
                    F.col(coluna),
                    F.lit(0)
                )
            )

    # ========================================================
    # 13. RENOMEAR COLUNAS PARA NEGÓCIO
    # ========================================================

    renomear = {

        # Comportamento
        "total_plays":
            "qtd_reproducoes_m0",

        "total_secs":
            "segundos_consumo_m0",

        "num_unq":
            "qtd_musicas_unicas_m0",

        "dias_ativos":
            "dias_uso_m0",

        # Financeiro
        "receita_mes":
            "receita_m0",

        "qtd_transacoes":
            "qtd_transacoes_m0",

        "cancelamentos_mes":
            "qtd_cancelamentos_m0",

        "is_auto_renew":
            "auto_renovacao_m0",

        "payment_plan_days":
            "duracao_plano_dias_m0",

        "plan_list_price":
            "preco_lista_m0"
    }

    for coluna_antiga, coluna_nova in renomear.items():

        if coluna_antiga in df_master.columns:

            df_master = df_master.withColumnRenamed(
                coluna_antiga,
                coluna_nova
            )

    # ========================================================
    # 14. RENOMEAR STATUS DO ASSINANTE
    # ========================================================

    if "is_ativo" in df_master.columns:

        df_master = df_master.withColumnRenamed(
            "is_ativo",
            "assinante_ativo_m0"
        )

    # ========================================================
    # 15. REMOVER DATAS TÉCNICAS
    # ========================================================
    #
    # A referência temporal da Master será somente:
    #   safra
    #
    # ========================================================

    colunas_tecnicas = [
        "safra_date_dt",
        "safra_date_m3",
        "registration_init_date",
        "is_ativo_m3"
    ]

    df_master = df_master.drop(
        *[
            c
            for c in colunas_tecnicas
            if c in df_master.columns
        ]
    )

    # ========================================================
    # 16. VALIDAÇÕES FINAIS
    # ========================================================

    print("\n[8] Validações finais...")

    # --------------------------------------------------------
    # Linhas
    # --------------------------------------------------------

    qtd_final = df_master.count()

    assert qtd_final == qtd_publico, (
        f"ERRO: Master possui {qtd_final:,} linhas, "
        f"mas deveria possuir {qtd_publico:,}."
    )

    print("✓ Quantidade de linhas preservada")

    # --------------------------------------------------------
    # Chave
    # --------------------------------------------------------

    validar_chave_unica(
        df_master,
        "MASTER"
    )

    # --------------------------------------------------------
    # Colunas duplicadas
    # --------------------------------------------------------

    validar_colunas_duplicadas(
        df_master,
        "MASTER"
    )

    # --------------------------------------------------------
    # Target
    # --------------------------------------------------------

    assert "churn_m3" in df_master.columns

    print(
        "✓ churn_m3 presente uma única vez"
    )

    # --------------------------------------------------------
    # Atividade M0
    # --------------------------------------------------------

    assert "assinante_ativo_m0" in df_master.columns

    print(
        "✓ assinante_ativo_m0 presente"
    )

    # --------------------------------------------------------
    # Safra
    # --------------------------------------------------------

    assert "safra" in df_master.columns

    print("✓ safra presente")

    # --------------------------------------------------------
    # Garantir ausência das datas técnicas
    # --------------------------------------------------------

    datas_restantes = [
        c
        for c in [
            "safra_date_dt",
            "safra_date_m3",
            "registration_init_date",
            "is_ativo_m3"
        ]
        if c in df_master.columns
    ]

    assert not datas_restantes, (
        f"ERRO: datas técnicas ainda presentes: "
        f"{datas_restantes}"
    )

    print("✓ Datas técnicas removidas")

    # ========================================================
    # 17. RESUMO
    # ========================================================

    print("\n" + "=" * 70)
    print("MASTER CONSTRUÍDA COM SUCESSO")
    print("=" * 70)

    print(
        f"Observações : {qtd_final:,}"
    )

    print(
        f"Clientes    : "
        f"{df_master.select(IDENTIFY).distinct().count():,}"
    )

    print(
        f"Features    : {len(df_master.columns)}"
    )

    # --------------------------------------------------------
    # Target
    # --------------------------------------------------------

    print("\nDistribuição do target:")

    (
        df_master
        .groupBy("churn_m3")
        .count()
        .orderBy("churn_m3")
        .show()
    )

    # --------------------------------------------------------
    # Uso
    # --------------------------------------------------------

    print("\nAusência de uso:")

    (
        df_master
        .groupBy("flag_sem_uso_m0")
        .count()
        .orderBy("flag_sem_uso_m0")
        .show()
    )

    # --------------------------------------------------------
    # Transação
    # --------------------------------------------------------

    print("\nAusência de transação:")

    (
        df_master
        .groupBy("flag_sem_transacao_m0")
        .count()
        .orderBy("flag_sem_transacao_m0")
        .show()
    )

    return df_master

In [12]:
%%time

members_path = (
    "/content/drive/MyDrive/SANTANDER/"
    "features_members.parquet"
)

logs_path = (
    "/content/drive/MyDrive/SANTANDER/"
    "features_logs.parquet"
)

transactions_path = (
    "/content/drive/MyDrive/SANTANDER/"
    "features_transactions.parquet"
)

output_path_final = (
    "/content/drive/MyDrive/SANTANDER/"
    "master.parquet"
)


# ============================================================
# CONSTRUÇÃO DA MASTER
# ============================================================

dfs_master = construir_master(
    members_path=members_path,
    logs_path=logs_path,
    transactions_path=transactions_path
)


CONSTRUÇÃO DA MASTER

[1] Lendo datasets...
Members      : 4,624,618
Logs         : 4,624,618
Transactions : 4,624,618

[2] Validando chave cliente × safra...
✓ Chave única validada: MEMBERS
✓ Chave única validada: LOGS
✓ Chave única validada: TRANSACTIONS

[3] Organizando colunas...

Colunas técnicas antes dos joins:
MEMBERS: ['churn_m3']
LOGS: []
TRANSACTIONS: []

[4] Criando Master a partir de Members...
Quantidade de observações do público: 4,624,618

[5] Adicionando comportamento de uso...
✓ Após Logs: 4,624,618 observações

[6] Adicionando informações financeiras...
✓ Após Transactions: 4,624,618 observações

[7] Criando métricas de rentabilidade...

[8] Validações finais...
✓ Quantidade de linhas preservada
✓ Chave única validada: MASTER
✓ Nenhuma coluna duplicada: MASTER
✓ churn_m3 presente uma única vez
✓ assinante_ativo_m0 presente
✓ safra presente
✓ Datas técnicas removidas

MASTER CONSTRUÍDA COM SUCESSO
Observações : 4,624,618
Clientes    : 665,041
Features    : 69

Distrib